In [ ]:
!pip install datasets

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 491.2/491.2 kB 23.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 116.3/116.3 kB 9.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 143.5/143.5 kB 11.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 194.8/194.8 kB 15.4 MB/s eta 0:00:00


In [ ]:
import transformers
from transformers import pipeline
from transformers import AutoTokenizer, AutoModelForSequenceClassification
import datasets
from datasets import load_dataset,DatasetDict
from transformers import TrainingArguments,Trainer

In [ ]:
tokenizer=AutoTokenizer.from_pretrained("google-bert/bert-base-multilingual-cased")

/usr/local/lib/python3.11/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


tokenizer_config.json:   0%|          | 0.00/49.0 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/625 [00:00<?, ?B/s]

vocab.txt:   0%|          | 0.00/996k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/1.96M [00:00<?, ?B/s]

In [ ]:
tokenizer

BertTokenizerFast(name_or_path='google-bert/bert-base-multilingual-cased', vocab_size=119547, model_max_length=512, is_fast=True, padding_side='right', truncation_side='right', special_tokens={'unk_token': '[UNK]', 'sep_token': '[SEP]', 'pad_token': '[PAD]', 'cls_token': '[CLS]', 'mask_token': '[MASK]'}, clean_up_tokenization_spaces=False, added_tokens_decoder={
	0: AddedToken("[PAD]", rstrip=False, lstrip=False, single_word=False, normalized=False, special=True),
	100: AddedToken("[UNK]", rstrip=False, lstrip=False, single_word=False, normalized=False, special=True),
	101: AddedToken("[CLS]", rstrip=False, lstrip=False, single_word=False, normalized=False, special=True),
	102: AddedToken("[SEP]", rstrip=False, lstrip=False, single_word=False, normalized=False, special=True),
	103: AddedToken("[MASK]", rstrip=False, lstrip=False, single_word=False, normalized=False, special=True),
}
)

In [ ]:
df=load_dataset("mounikaiiith/Telugu_Sentiment")

README.md:   0%|          | 0.00/921 [00:00<?, ?B/s]

Sentiment_train.csv:   0%|          | 0.00/5.94M [00:00<?, ?B/s]

Sentiment_valid%20.csv:   0%|          | 0.00/848k [00:00<?, ?B/s]

Sentiment_test.csv:   0%|          | 0.00/1.70M [00:00<?, ?B/s]

Generating train split:   0%|          | 0/24599 [00:00<?, ? examples/s]

Generating validation split:   0%|          | 0/3510 [00:00<?, ? examples/s]

Generating test split:   0%|          | 0/7033 [00:00<?, ? examples/s]

In [ ]:
df

DatasetDict({
    train: Dataset({
        features: ['Unnamed: 0', 'Sentence', 'Sentiment'],
        num_rows: 24599
    })
    validation: Dataset({
        features: ['Unnamed: 0', 'Sentence', 'Sentiment'],
        num_rows: 3510
    })
    test: Dataset({
        features: ['Unnamed: 0', 'Sentence', 'Sentiment'],
        num_rows: 7033
    })
})

In [ ]:
from datasets import DatasetDict

# Remove 'Unnamed: 0' from all splits
df = DatasetDict({
    split: dataset.remove_columns('Unnamed: 0')
    for split, dataset in df.items()
})


In [ ]:
df

DatasetDict({
    train: Dataset({
        features: ['Sentence', 'Sentiment'],
        num_rows: 24599
    })
    validation: Dataset({
        features: ['Sentence', 'Sentiment'],
        num_rows: 3510
    })
    test: Dataset({
        features: ['Sentence', 'Sentiment'],
        num_rows: 7033
    })
})

In [ ]:
set(df['train']['Sentiment'])

{'neg', 'neutral', 'pos'}

In [ ]:
from datasets import ClassLabel

In [ ]:
cl = ClassLabel(num_classes=3, names=["neutral", "pos", "neg"])

In [ ]:
def pre_pro(batch):
  # feature variables
  l=[]
  for y in batch["Sentence"]:
      if y is not None:
         l.append(str(y))
      else:
        l.append("")
  tokenizers=tokenizer(l,max_length=512,truncation=True,padding="max_length")

  # class variables
  l1=[]
  for y1 in batch["Sentiment"]:
    if y1 is None:
      l1.append(0)
    else:
      l1.append(cl.str2int(y1))

  tokenizers["label"]=l1

  return tokenizers

In [ ]:
final_data=df.map(pre_pro,batched=True,remove_columns=["Sentence","Sentiment"])

Map:   0%|          | 0/24599 [00:00<?, ? examples/s]

Map:   0%|          | 0/3510 [00:00<?, ? examples/s]

Map:   0%|          | 0/7033 [00:00<?, ? examples/s]

In [ ]:
final_data

DatasetDict({
    train: Dataset({
        features: ['input_ids', 'token_type_ids', 'attention_mask', 'label'],
        num_rows: 24599
    })
    validation: Dataset({
        features: ['input_ids', 'token_type_ids', 'attention_mask', 'label'],
        num_rows: 3510
    })
    test: Dataset({
        features: ['input_ids', 'token_type_ids', 'attention_mask', 'label'],
        num_rows: 7033
    })
})

In [ ]:
model=AutoModelForSequenceClassification.from_pretrained("google-bert/bert-base-multilingual-cased",num_labels=3)

model.safetensors:   0%|          | 0.00/714M [00:00<?, ?B/s]

Some weights of BertForSequenceClassification were not initialized from the model checkpoint at google-bert/bert-base-multilingual-cased and are newly initialized: ['classifier.bias', 'classifier.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


In [ ]:
model

BertForSequenceClassification(
  (bert): BertModel(
    (embeddings): BertEmbeddings(
      (word_embeddings): Embedding(119547, 768, padding_idx=0)
      (position_embeddings): Embedding(512, 768)
      (token_type_embeddings): Embedding(2, 768)
      (LayerNorm): LayerNorm((768,), eps=1e-12, elementwise_affine=True)
      (dropout): Dropout(p=0.1, inplace=False)
    )
    (encoder): BertEncoder(
      (layer): ModuleList(
        (0-11): 12 x BertLayer(
          (attention): BertAttention(
            (self): BertSdpaSelfAttention(
              (query): Linear(in_features=768, out_features=768, bias=True)
              (key): Linear(in_features=768, out_features=768, bias=True)
              (value): Linear(in_features=768, out_features=768, bias=True)
              (dropout): Dropout(p=0.1, inplace=False)
            )
            (output): BertSelfOutput(
              (dense): Linear(in_features=768, out_features=768, bias=True)
              (LayerNorm): LayerNorm((768,), eps=1

In [ ]:
ta=TrainingArguments(eval_strategy ="epoch",
                  learning_rate=2e-5,save_strategy="epoch",
                  logging_dir="/content/models_save",
                  num_train_epochs=3,output_dir="/content/models_save",
                  per_device_train_batch_size=8,per_device_eval_batch_size=8,
                  metric_for_best_model="eval_loss",fp16=True,load_best_model_at_end=True)

In [ ]:
ta

TrainingArguments(
_n_gpu=1,
accelerator_config={'split_batches': False, 'dispatch_batches': None, 'even_batches': True, 'use_seedable_sampler': True, 'non_blocking': False, 'gradient_accumulation_kwargs': None, 'use_configured_state': False},
adafactor=False,
adam_beta1=0.9,
adam_beta2=0.999,
adam_epsilon=1e-08,
auto_find_batch_size=False,
average_tokens_across_devices=False,
batch_eval_metrics=False,
bf16=False,
bf16_full_eval=False,
data_seed=None,
dataloader_drop_last=False,
dataloader_num_workers=0,
dataloader_persistent_workers=False,
dataloader_pin_memory=True,
dataloader_prefetch_factor=None,
ddp_backend=None,
ddp_broadcast_buffers=None,
ddp_bucket_cap_mb=None,
ddp_find_unused_parameters=None,
ddp_timeout=1800,
debug=[],
deepspeed=None,
disable_tqdm=False,
dispatch_batches=None,
do_eval=True,
do_predict=False,
do_train=False,
eval_accumulation_steps=None,
eval_delay=0,
eval_do_concat_batches=True,
eval_on_start=False,
eval_steps=None,
eval_strategy=IntervalStrategy.EPOCH,
eval_

In [ ]:
from transformers import TrainingArguments, Trainer, EarlyStoppingCallback
tr=Trainer(model=model,
        args=ta,
        train_dataset=final_data["train"],
        eval_dataset=final_data["validation"],
        tokenizer=tokenizer,
        callbacks=[EarlyStoppingCallback(early_stopping_patience=2)])

<ipython-input-25-555d9f31a947>:2: FutureWarning: `tokenizer` is deprecated and will be removed in version 5.0.0 for `Trainer.__init__`. Use `processing_class` instead.
  tr=Trainer(model=model,


In [ ]:
model = tr.train()

Epoch,Training Loss,Validation Loss
1,0.797500,0.753976
2,0.681100,0.723404
3,0.550400,0.769553


In [ ]:
# from google.colab import files
# files.download("models_save.zip")

In [ ]:
# Save the trained model, not the TrainOutput object
tr.model.save_pretrained("/content/telugu_bert_model_new")
tokenizer.save_pretrained("/content/telugu_bert_model_new")

('/content/telugu_bert_model_new/tokenizer_config.json',
 '/content/telugu_bert_model_new/special_tokens_map.json',
 '/content/telugu_bert_model_new/vocab.txt',
 '/content/telugu_bert_model_new/added_tokens.json',
 '/content/telugu_bert_model_new/tokenizer.json')

In [ ]:
from transformers import AutoModelForSequenceClassification, AutoTokenizer

# Load model directly from the path where you saved it
model = AutoModelForSequenceClassification.from_pretrained("/content/telugu_bert_model_new")
tokenizer = AutoTokenizer.from_pretrained("/content/telugu_bert_model_new")

In [ ]:
!pip install -q huggingface_hub
from huggingface_hub import notebook_login

notebook_login()

In [ ]:
model.push_to_hub("Gowthamvemula/Teugu_Sentimental_fine-tuning")
tokenizer.push_to_hub("Gowthamvemula/Teugu_Sentimental_fine-tuning")

model.safetensors:   0%|          | 0.00/711M [00:00<?, ?B/s]

README.md:   0%|          | 0.00/5.17k [00:00<?, ?B/s]

CommitInfo(commit_url='https://huggingface.co/Gowthamvemula/Teugu_Sentimental_fine-tuning/commit/f7bc098975c49a32719552a457f48b3a144f9e0d', commit_message='Upload tokenizer', commit_description='', oid='f7bc098975c49a32719552a457f48b3a144f9e0d', pr_url=None, repo_url=RepoUrl('https://huggingface.co/Gowthamvemula/Teugu_Sentimental_fine-tuning', endpoint='https://huggingface.co', repo_type='model', repo_id='Gowthamvemula/Teugu_Sentimental_fine-tuning'), pr_revision=None, pr_num=None)

In [ ]:
# Use a pipeline as a high-level helper
from transformers import pipeline

pipe = pipeline("text-classification", model="Gowthamvemula/Teugu_Sentimental_fine-tuning")

/usr/local/lib/python3.11/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


config.json:   0%|          | 0.00/1.05k [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/711M [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/1.41k [00:00<?, ?B/s]

vocab.txt:   0%|          | 0.00/996k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/2.92M [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/695 [00:00<?, ?B/s]

Device set to use cuda:0


In [ ]:
# Load model directly
from transformers import AutoTokenizer, AutoModelForSequenceClassification

tokenizer = AutoTokenizer.from_pretrained("Gowthamvemula/Telugu_Sentimental_Analysis")
model = AutoModelForSequenceClassification.from_pretrained("Gowthamvemula/Telugu_Sentimental_Analysis")

In [ ]:

# Use a pipeline as a high-level helper
from transformers import pipeline

pipe = pipeline("text-classification", model="Gowthamvemula/Teugu_Sentimental_fine-tuning")
# Example usage:
test_text = "అన్ని టెండర్లూ పారదర్శకంగా ఉంటాయనీ, ఎల్లో మీడియా దురుద్దేశంపూర్వకంగా వార్తలు రాస్తే కచ్చితంగా శిక్షిస్తామని, పరువునష్టం దావా వేస్తామన్నారు."  # Example Telugu text
result = pipe(test_text)
result[0]

idx = int(result[0]['label'].split('_')[1])

if idx == 0:
  print("Neutral")
elif idx==1:
  print("Positive")
else:
  print("Negative")

{'label': 'LABEL_1', 'score': 0.5370363593101501}

In [ ]:
idx = int(result[0]['label'].split('_')[1])

In [ ]:
names=["neutral", "pos", "neg"]

pi

In [ ]:
# prompt: give me some 10 to 20 give me some more examples

# Example usage with different Telugu texts:
test_texts = [
    #  "ఈ సినిమా చాలా బాగుంది",  # This movie is very good
    # "నేను ఈ పుస్తకాన్ని చాలా ఇష్టపడ్డాను",  # I liked this book very much
     "ఈ ఆహారం చాలా చెడుగా ఉంది",  # This food is very bad
    "నాకు ఈ రోజు చాలా సంతోషంగా ఉంది",  # I am very happy today
     "నేను ఈ వార్తలకు చాలా బాధపడ్డాను", # I felt very sad for this news
     "ఈ వాతావరణం చాలా అద్భుతంగా ఉంది", # This weather is very wonderful
     "నేను ఈ పనిని పూర్తి చేయలేకపోతున్నాను", # I can not complete this work
     "ఈ కారు చాలా వేగంగా ఉంది",  # This car is very fast
     "ఈ పాట చాలా అందంగా ఉంది",  # This song is very beautiful
     "ఈ ప్రదేశం చాలా ప్రశాంతంగా ఉంది", # This place is very calm
    # "నేను చాలా అలసిపోయాను", # I am very tired
    #  "ఈ సమస్యకు పరిష్కారం లేదు", # There is no solution to this problem
     "నేను ఈ సమాచారాన్ని నమ్మలేకపోతున్నాను", # I can not believe this information
     "ఈ వ్యక్తి చాలా మంచివాడు", # This person is very good
     "ఈ పనిని త్వరగా పూర్తి చేయాలి" #This work should be completed quickly
    # # # Add more examples here...
]

for text in test_texts:
    result = pipe(text)
    print(f"Text: {text}")
    print(f"Result: {result}")
    print("-" * 20)


You seem to be using the pipelines sequentially on GPU. In order to maximize efficiency please use a dataset


Text: ఈ ఆహారం చాలా చెడుగా ఉంది
Result: [{'label': 'LABEL_0', 'score': 0.711139976978302}]
--------------------
Text: నాకు ఈ రోజు చాలా సంతోషంగా ఉంది
Result: [{'label': 'LABEL_1', 'score': 0.9051015377044678}]
--------------------
Text: నేను ఈ వార్తలకు చాలా బాధపడ్డాను
Result: [{'label': 'LABEL_2', 'score': 0.9480864405632019}]
--------------------
Text: ఈ వాతావరణం చాలా అద్భుతంగా ఉంది
Result: [{'label': 'LABEL_1', 'score': 0.9276691675186157}]
--------------------
Text: నేను ఈ పనిని పూర్తి చేయలేకపోతున్నాను
Result: [{'label': 'LABEL_0', 'score': 0.5183483958244324}]
--------------------
Text: ఈ కారు చాలా వేగంగా ఉంది
Result: [{'label': 'LABEL_1', 'score': 0.8319589495658875}]
--------------------
Text: ఈ పాట చాలా అందంగా ఉంది
Result: [{'label': 'LABEL_1', 'score': 0.8192635774612427}]
--------------------
Text: ఈ ప్రదేశం చాలా ప్రశాంతంగా ఉంది
Result: [{'label': 'LABEL_1', 'score': 0.5367408394813538}]
--------------------
Text: నేను ఈ సమాచారాన్ని నమ్మలేకపోతున్నాను
Result: [{'label': 'LABEL_2'